<a href="https://colab.research.google.com/github/Samlat-stack/house-price-prediction-in-california/blob/main/California_house_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [109]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [110]:
df.shape


(20640, 9)

In [111]:
df["MedHouseVal"].value_counts()

,count
MedHouseVal,
5.00001,965
1.37500,122
1.62500,117
1.12500,103
1.87500,93
...,...
0.34200,1
0.46200,1
3.52000,1


In [112]:
df.isna().sum()

,0
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0
MedHouseVal,0


In [113]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [114]:
df[df["AveRooms"] > 20]

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
1024,3.1500,16.0,29.852941,5.323529,202.0,1.980392,38.52,-120.00,1.406
1102,2.4028,17.0,31.777778,9.703704,47.0,1.740741,40.06,-121.54,0.675
1234,4.4063,13.0,28.757282,5.300971,252.0,2.446602,38.26,-120.33,1.558
1238,3.3542,10.0,28.112903,5.604839,298.0,2.403226,38.29,-120.27,1.038
1239,3.0750,13.0,33.989691,6.824742,267.0,2.752577,38.31,-120.27,1.083
...,...,...,...,...,...,...,...,...,...
19736,6.1359,14.0,22.823529,6.352941,35.0,2.058824,40.12,-121.78,1.063
19975,2.1029,11.0,23.994152,5.157895,356.0,2.081871,36.12,-118.54,0.991
19976,3.1250,10.0,21.095238,5.285714,48.0,2.285714,36.19,-118.37,0.713
20093,2.0625,37.0,24.669118,4.897059,338.0,2.485294,38.12,-120.12,0.889


In [115]:
print(df.shape)                   # before
df = df[df["AveRooms"] <= 20]
print(df.shape)                   # after

(20640, 9)
(20571, 9)


In [116]:
X = df.drop("MedHouseVal", axis=1)   # clues: everything except the price
y = df["MedHouseVal"]                # answer: the price

print(X.shape, y.shape)
print(X.columns.tolist())

(20571, 8) (20571,)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [117]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(16456, 8) (4115, 8)


In [118]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# the dumb baseline: always guess the average training price
dumb_preds = [y_train.mean()] * len(y_test)
print("Dumb MAE:", round(mean_absolute_error(y_test, dumb_preds), 3))

# the student studies, then guesses the hidden prices
reg = LinearRegression()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)

print("MAE: ", round(mean_absolute_error(y_test, preds), 3))
print("RMSE:", round(mean_squared_error(y_test, preds) ** 0.5, 3))
print("R2:  ", round(reg.score(X_test, y_test), 3))

Dumb MAE: 0.915
MAE:  0.526
RMSE: 0.712
R2:   0.617


In [119]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("MAE: ", round(mean_absolute_error(y_test, rf_preds), 3))
print("RMSE:", round(mean_squared_error(y_test, rf_preds) ** 0.5, 3))
print("R2:  ", round(rf.score(X_test, y_test), 3))

MAE:  0.327
RMSE: 0.501
R2:   0.811


In [ ]:
from sklearn.model_selection import cross_val_score

models = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(random_state=42, n_jobs=-1),
}

for name, m in models.items():
    scores = -cross_val_score(m, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE average:", round(scores.mean(), 3), "| spread:", round(scores.std(), 3))

Linear regression MAE average: 0.542 | spread: 0.021


In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, m in models.items():
    scores = -cross_val_score(m, X, y, cv=kf, scoring="neg_mean_absolute_error")
    print(name, "MAE average:", round(scores.mean(), 3), "| spread:", round(scores.std(), 3))

# California House Price Prediction (Regression)

## Goal
Predict the median house value of an area in California from census
features such as median income, house age, and location.
Prices are in units of $100,000.

## Data and cleaning
- Dataset: California Housing (from scikit-learn), 20,640 areas, 8 features.
- No missing values and no text columns.
- `AveRooms` had extreme outliers (max 141.9 vs. mean 5.4). I dropped the
  69 areas with more than 20 average rooms (about 0.3% of the data),
  because they were rare and could distort the model. This is a judgment
  call and could be tested by keeping them.
- Data was split 80% training and 20% testing.

## Results (MAE, lower is better; 0.1 = $10,000)
| Model                          | Test MAE | Test R² |
|--------------------------------|----------|---------|
| Dumb baseline (always average) | 0.915    | -       |
| Linear regression              | 0.526    | 0.617   |
| Random forest                  | 0.327    | 0.811   |

## Cross-validation (5-fold, MAE)
| Model             | Unshuffled | Shuffled |
|-------------------|------------|----------|
| Linear regression | 0.542      | 0.526    |
| Random forest     | 0.463      | 0.327    |

Shuffled folds test on areas similar to ones the model has studied.
Unshuffled folds seem to be closer to testing on unseen regions, because
the data appears to be ordered by location (inferred from the results,
not checked directly). For use in unseen parts of California, the
unshuffled score is the more realistic one.

## What I learned
- MAE is an error, so lower is better (the opposite of accuracy).
- The random forest beat linear regression clearly in both setups.
- How you split the data changes your score, so I report both.


## Next steps
- Test whether keeping the 69 outliers changes the score.
- Hold out whole regions (by latitude/longitude) for a stricter test.
- Try gradient boosting.